# typesafe

> Ask TypeSafe's Jev model yes/no questions about text

In [ ]:
#| default_exp typesafe

In [ ]:
#| hide
from nbdev.showdoc import *

TypeSafe's [Jev](https://docs.typesafe.ai) model reads a `state` and answers typed questions about it with probabilities. It generates no text. This module calls its HTTP API. `noul` builds a yes/no question, `systemone` sends one request, and `ask_nouls` returns the probability of yes for each question.

The functions read the API key from `TYPESAFE_API_KEY`. A request sends its `state` and questions to TypeSafe's servers and costs $0.042 per million input tokens.

In [ ]:
#| export
import time, httpx2
from fastcore.utils import *
from fastcore.meta import delegates

In [ ]:
from fastcore.test import *

## Questions

A Noul is TypeSafe's name for a yes/no question. Jev answers it with the probability that the answer is yes. `instructions` holds the question. It can be a string, or a dict that holds the question beside the data it asks about. `true` and `false` say what counts as each answer. Each can be a string, a dict or a list.

In [ ]:
#| export
def noul(
    instructions, # The yes/no question: a string, or a dict or list that holds the question beside its data
    true=None, # What counts as yes: a string, dict or list
    false=None, # What counts as no: a string, dict or list
):
    "A Noul question, which Jev answers with the probability of yes"
    q = dict(type='noul', instructions=instructions)
    if true is not None or false is not None: q['criteria'] = {'true': true, 'false': false}
    return q

A question with no criteria has two fields:

In [ ]:
q = noul('Is this message about billing?')
test_eq(q, dict(type='noul', instructions='Is this message about billing?'))
q

A dict keeps the text being judged apart from the question about it. The question names the field in backticks:

In [ ]:
q = noul(dict(message='How long do refunds take?', question='Is `message` a refund request?'),
    true='The customer asks for their money back.', false='The customer asks about the refund policy.')
test_eq(q['criteria'], {'true': 'The customer asks for their money back.', 'false': 'The customer asks about the refund policy.'})
q

## Requests

`systemone` posts one request. Jev reads `state` once and answers every question against it. Put questions that share a state in one request. A request must fit 64,000 tokens in total. `state` plus any one question must fit 32,000.

`systemone` waits and tries again after a 429 response, which means the rate limit was reached, or a 529, which means TypeSafe is overloaded. Any other error raises `RuntimeError` with TypeSafe's explanation.

In [ ]:
#| export
TYPESAFE_URL = 'https://api.typesafe.ai'

def systemone(
    state, # Content that every question refers to: a string, dict or list
    questions, # Question name -> question, such as `noul` returns
    model='jev-1.14.0', # Model ID or alias, such as `jev-latest`
    timeout=120, # Seconds to wait for each response
    retries=4, # Extra attempts after a 429 or 529 response
):
    "TypeSafe's response to `questions` about `state`"
    hdrs = {'Authorization': f"Bearer {os.environ['TYPESAFE_API_KEY']}"}
    for i in range(retries+1):
        r = httpx2.post(f'{TYPESAFE_URL}/v1/systemone', headers=hdrs, timeout=timeout,
            json=dict(model=model, state=state, questions=questions))
        if r.status_code not in (429, 529) or i == retries: break
        time.sleep(2**i)
    if r.is_error: raise RuntimeError(f'TypeSafe {r.status_code}: {r.text}')
    return r.json()

@delegates(systemone)
def ask_nouls(state, questions, **kwargs):
    "The probability of yes for each Noul in `questions`"
    return {k: v['noul'] for k,v in systemone(state, questions, **kwargs)['answers'].items()}

The response names the model that answered and counts the tokens used. The cell needs `TYPESAFE_API_KEY` and network access, so the docs build does not run it:

In [ ]:
#| eval: false
msg = 'I was charged twice for my subscription this month. Please refund the extra charge.'
res = systemone(msg, dict(billing=noul('Is this message about billing?'), greeting=noul('Is this message a greeting?')))
res

`ask_nouls` keeps only the probabilities:

In [ ]:
#| eval: false
ask_nouls(msg, dict(billing=noul('Is this message about billing?'), greeting=noul('Is this message a greeting?')))

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()